what’s the difference between `Interface` and `Blocks`?

-   ⚡ `Interface`: a high-level API that allows you to create a full machine learning demo simply by providing a list of inputs and outputs.
-   🧱 `Blocks`: a low-level API that allows you to have full control over the data flows and layout of your application. You can build very complex, multi-step applications using `Blocks` (as in “building blocks”).

The `Interface` API is extremely easy to use but lacks the flexibility that the `Blocks` API provides. For example, you might want to:

-   Group together related demos as multiple tabs in one web application
-   Change the layout of your demo, e.g. to specify where the inputs and outputs are located
-   Have multi-step interfaces, in which the output of one model becomes the input to the next model, or have more flexible data flows in general
-   Change a component’s properties (for example, the choices in a dropdown) or its visibility based on user input

# Creating a simple demo using Blocks

In [ ]:
import gradio as gr


def flip_text(x):
    return x[::-1]


demo = gr.Blocks()

with demo:
    gr.Markdown(
        """
    # Flip Text!
    Start typing below to see the output.
    """
    )
    input = gr.Textbox(placeholder="Flip this text")
    output = gr.Textbox()

    input.change(fn=flip_text, inputs=input, outputs=output)

demo.launch()

This simple example above introduces 4 concepts that underlie Blocks:

1.  Blocks allow you to build web applications that combine markdown, HTML, buttons, and interactive components simply by instantiating objects in Python inside of a `with gradio.Blocks` context.   
2.  You can define regular Python functions anywhere in your code and run them with user input using `Blocks`.
    - In our example, we have a simple function that “flips” the input text, but you can write any Python function, from a simple calculation to processing the predictions from a machine learning model.
3.  You can assign events to any `Blocks` component. This will run your function when the component is clicked, changed, etc.
    - When you assign an event, you pass in three parameters:
        - `fn`: the function that should be called,
        - `inputs`: the (list) of input component(s), and
        - `outputs`: the (list) of output components that should be called.
    - In the example above, we run the `flip_text()` function when the value in the `Textbox` named input `input` changes. The event reads the value in `input`, passes it as the name parameter to `flip_text()`, which then returns a value that gets assigned to our second `Textbox` named `output`.
    - To see a list of events that each component supports, see the Gradio [documentation](https://www.gradio.app/docs/).
4.  Blocks automatically figures out whether a component should be interactive (accept user input) or not, based on the event triggers you define.
    - In our example, the first textbox is interactive, since its value is used by the `flip_text()` function.
    - The second textbox is not interactive, since its value is never used as an input.
    - In some cases, you might want to override this, which you can do by passing a boolean to the `interactive` parameter of the component
        - (e.g. `gr.Textbox(placeholder="Flip this text", interactive=True)`).

# Customizing the layout of your demo

1. By default, `Blocks` renders the components that you create vertically in one column.
2. You can change that by creating additional columns `with gradio.Column():` or rows `with gradio.Row():` and creating components within those contexts.
    - Any components created under a `Column` (this is also the default) will be laid out vertically.
    - Any component created under a `Row` will be laid out horizontally
    - similar to the [flexbox model in web development](https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_Flexible_Box_Layout/Basic_Concepts_of_Flexbox).

3. You can also create tabs for your demo by using the `with gradio.Tabs()` context manager.
    - Within this context, you can create multiple tabs by specifying `with gradio.TabItem(name_of_tab):` children.
    - Any component created inside of a `with gradio.TabItem(name_of_tab):` context appears in that tab.

In [ ]:
import numpy as np
import gradio as gr

demo = gr.Blocks()


def flip_text(x):
    return x[::-1]


def flip_image(x):
    return np.fliplr(x)


with demo:
    gr.Markdown("Flip text or image files using this demo.")
    with gr.Tabs():
        with gr.TabItem("Flip Text"):
            with gr.Row():
                text_input = gr.Textbox()
                text_output = gr.Textbox()
            text_button = gr.Button("Flip")
        with gr.TabItem("Flip Image"):
            with gr.Row():
                image_input = gr.Image()
                image_output = gr.Image()
            image_button = gr.Button("Flip")

    text_button.click(flip_text, inputs=text_input, outputs=text_output)
    image_button.click(flip_image, inputs=image_input, outputs=image_output)

demo.launch()

In [ ]:
demo.close()

# Exploring events and state

1. Just as you can control the layout, `Blocks` gives you fine-grained control over what events trigger function calls. Each component and many layouts have specific events that they support.
2. For example, the `Textbox` component has 2 events: `change()` (when the value inside of the textbox changes), and `submit()` (when a user presses the enter key while focused on the textbox).
3. More complex components can have even more events: for example, the `Audio` component also has separate events for when the audio file is played, cleared, paused, etc.
4. You can attach event trigger to none, one, or more of these events. You create an event trigger by calling the name of the event on the component instance as a function — e.g. `textbox.change(...)` or `btn.click(...)`. The function takes in three parameters, as discussed above:
    -   `fn`: the function to run
    -   `inputs`: a (list of) component(s) whose values should supplied as the input parameters to the function. Each component’s value gets mapped to the corresponding function parameter, in order. This parameter can be None if the function does not take any parameters.
    -   `outputs`: a (list of) component(s) whose values should be updated based on the values returned by the function. Each return value sets the corresponding component’s value, in order. This parameter can be None if the function does not return anything.
5. You can even make the input and output component be the same component, as below:

In [ ]:
import gradio as gr

def complete_with_mock_gpt(text):
    # Use the last 50 characters of the text as context
    return text + ">>>> the completion"


with gr.Blocks() as demo:
    textbox = gr.Textbox(placeholder="Type here and press enter...", lines=4)
    btn = gr.Button("Generate")

    btn.click(complete_with_mock_gpt, textbox, textbox)

demo.launch()

In [ ]:
demo.close()

# Creating multi-step demos

In [ ]:
from transformers import pipeline
import gradio as gr

asr = pipeline("automatic-speech-recognition", "facebook/wav2vec2-base-960h")
classifier = pipeline("text-classification")

In [ ]:
def speech_to_text(speech):
    text = asr(speech)["text"]
    return text

def text_to_sentiment(text):
    return classifier(text)[0]["label"]

In [ ]:
demo = gr.Blocks()

with demo:
    audio_file = gr.Audio(type="filepath")
    text = gr.Textbox()
    label = gr.Label()

    b1 = gr.Button("Recognize Speech")
    b2 = gr.Button("Classify Sentiment")

    b1.click(speech_to_text, inputs=audio_file, outputs=text)
    b2.click(text_to_sentiment, inputs=text, outputs=label)

demo.launch()

In [ ]:
demo.close()

# Updating Component Properties

- what happens if you want to change other properties of a component, like the visibility of a textbox or the choices in a radio button group?
- You can do this by returning a component class’s `update()` method instead of a regular return value from your function.

In [ ]:
import gradio as gr


def change_textbox(choice):
    if choice == "short":
        return gr.Textbox(lines=2, visible=True)
    elif choice == "long":
        return gr.Textbox(lines=8, visible=True)
    else:
        return gr.Textbox(visible=False)


with gr.Blocks() as block:
    radio = gr.Radio(
        ["short", "long", "none"], label="What kind of essay would you like to write?"
    )
    text = gr.Textbox(lines=2, interactive=True)

    radio.change(fn=change_textbox, inputs=radio, outputs=text)
    block.launch()

In [ ]:
demo.close()